In [1]:
import string
import nltk
nltk.download("stopwords")
nltk.download("punkt")

# used for looping through folders/files
from os import listdir
from os.path import isfile, join

#Calc tfidf and cosine similarity
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Lenovo\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\Lenovo\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!


In [2]:
# All text entries to compare will appear here
BASE_INPUT_DIR = "Cosine"

## Preprocess Data

#### File information

In [3]:
def returnListOfFilePaths(folderPath):
    fileInfo = []
    listOfFileNames = [fileName for fileName in listdir(folderPath) if isfile(join(folderPath, fileName))]
    listOfFilePaths = [join(folderPath, fileName) for fileName in listdir(folderPath) if isfile(join(folderPath, fileName))]
    fileInfo.append(listOfFileNames)
    fileInfo.append(listOfFilePaths)
    return fileInfo

fileNames, filePaths = returnListOfFilePaths(BASE_INPUT_DIR)
print(fileNames, "\n", filePaths)

['f1.txt', 'f2.txt', 'f3.txt'] 
 ['Cosine\\f1.txt', 'Cosine\\f2.txt', 'Cosine\\f3.txt']


In [4]:
# Get document contents
def create_docContentDict(filePaths):
    rawContentDict = {}
    for filePath in filePaths:
        with open(filePath, "r") as ifile:
            fileContent = ifile.read()
        rawContentDict[filePath] = fileContent
    return rawContentDict
rawContentDict = create_docContentDict(filePaths)
print(rawContentDict)

{'Cosine\\f1.txt': 'Channel tunnel operator Eurotunnel on Monday announced details of a deal giving bank creditors 45.5 percent of the company in return for wiping out 1.0 billion pounds ($1.6 billion) of its massive debts.\nThe long-awaited but highly complex restructuring of nearly nearly nine billion pounds of debt and unpaid interest throws the company a lifeline which could secure what is still likely to be a difficult future.\nThe deal, announced simultaneously in Paris and London, brings the company back from the brink of bankruptcy but leaves current shareholders, who have already seen their investment dwindle, owning only 54.5 percent of the company.\n"We have fixed and capped the interest payments and arranged only to pay what is available in cash," Eurotunnel co-chairman Alastair Morton told reporters at a news conference. "Avoiding having to do this again is the name of the game."\nMorton said the plan provides the Anglo-French company with the medium term financial stabili

## Create Custom tokenizer

### Define functions to use within the tokenizer
We'd like to;
- tokenize the input
- remove stop words
- perform stemming
- remove punctuation
- convert input to lowercase

#### Tokenize

In [5]:
def tokenizeContent(contentsRaw):
    tokenized = nltk.tokenize.word_tokenize(contentsRaw)
    return tokenized

#### Remove Stop words

In [6]:
def removeStopWordsFromTokenized(contentsTokenized):
    stop_word_set = set(nltk.corpus.stopwords.words("english"))
    filteredContents = [word for word in contentsTokenized if word not in stop_word_set]
    return filteredContents

#### Stemming

In [7]:
def performPorterStemmingOnContents(contentsTokenized):
    porterStemmer = nltk.stem.PorterStemmer()
    filteredContents = [porterStemmer.stem(word) for word in contentsTokenized]
    return filteredContents

#### Remove Punctuation

In [8]:
def removePunctuationFromTokenized(contentsTokenized):
    excludePuncuation = set(string.punctuation)

    # manually add additional punctuation to remove
    doubleSingleQuote = '\'\''
    doubleDash = '--'
    doubleTick = '``'

    excludePuncuation.add(doubleSingleQuote)
    excludePuncuation.add(doubleDash)
    excludePuncuation.add(doubleTick)

    filteredContents = [word for word in contentsTokenized if word not in excludePuncuation]
    return filteredContents

#### Convert terms to lowercase

In [9]:
def convertItemsToLower(contentsRaw):
    filteredContents = [term.lower() for term in contentsRaw]
    return filteredContents

### Test that functions are working as expected

In [10]:
# get contents of a file for testing
# TODO: may need to make a copy of this here
content_test = rawContentDict[filePaths[0]]

# visually inspect
print(content_test[:300])

Channel tunnel operator Eurotunnel on Monday announced details of a deal giving bank creditors 45.5 percent of the company in return for wiping out 1.0 billion pounds ($1.6 billion) of its massive debts.
The long-awaited but highly complex restructuring of nearly nearly nine billion pounds of debt a


In [11]:
# test tokenization
content_test_tokenized = tokenizeContent(content_test)

# visually inspect
print(content_test_tokenized[:30])

['Channel', 'tunnel', 'operator', 'Eurotunnel', 'on', 'Monday', 'announced', 'details', 'of', 'a', 'deal', 'giving', 'bank', 'creditors', '45.5', 'percent', 'of', 'the', 'company', 'in', 'return', 'for', 'wiping', 'out', '1.0', 'billion', 'pounds', '(', '$', '1.6']


In [12]:
# test remove stop words
content_test_rmStop = removeStopWordsFromTokenized(content_test_tokenized)

# visually inspect
print(content_test_rmStop[:30])

['Channel', 'tunnel', 'operator', 'Eurotunnel', 'Monday', 'announced', 'details', 'deal', 'giving', 'bank', 'creditors', '45.5', 'percent', 'company', 'return', 'wiping', '1.0', 'billion', 'pounds', '(', '$', '1.6', 'billion', ')', 'massive', 'debts', '.', 'The', 'long-awaited', 'highly']


In [13]:
# Test stemming
content_test_stemmed = performPorterStemmingOnContents(content_test_rmStop)

# visually inspect
print(content_test_stemmed[:30])

['channel', 'tunnel', 'oper', 'eurotunnel', 'monday', 'announc', 'detail', 'deal', 'give', 'bank', 'creditor', '45.5', 'percent', 'compani', 'return', 'wipe', '1.0', 'billion', 'pound', '(', '$', '1.6', 'billion', ')', 'massiv', 'debt', '.', 'the', 'long-await', 'highli']


In [14]:
# Test remove punctuation
content_test_cleaned = removePunctuationFromTokenized(content_test_stemmed)

# visually inspect
print(content_test_cleaned[:30])

['channel', 'tunnel', 'oper', 'eurotunnel', 'monday', 'announc', 'detail', 'deal', 'give', 'bank', 'creditor', '45.5', 'percent', 'compani', 'return', 'wipe', '1.0', 'billion', 'pound', '1.6', 'billion', 'massiv', 'debt', 'the', 'long-await', 'highli', 'complex', 'restructur', 'nearli', 'nearli']


In [15]:
# Test convert to lower
content_test_clean_lower = convertItemsToLower(content_test_cleaned)
print(content_test_clean_lower[:30])

['channel', 'tunnel', 'oper', 'eurotunnel', 'monday', 'announc', 'detail', 'deal', 'give', 'bank', 'creditor', '45.5', 'percent', 'compani', 'return', 'wipe', '1.0', 'billion', 'pound', '1.6', 'billion', 'massiv', 'debt', 'the', 'long-await', 'highli', 'complex', 'restructur', 'nearli', 'nearli']


### Wrap into a function to be used by NLTK

In [16]:
# process data without writing inspection file information to file
def processData(rawContents):
    cleaned = tokenizeContent(rawContents)
    cleaned = removeStopWordsFromTokenized(cleaned)
    cleaned = performPorterStemmingOnContents(cleaned)
    cleaned = removePunctuationFromTokenized(cleaned)
    cleaned = convertItemsToLower(cleaned)
    return cleaned

## Create Functions For Output
- TFIDF
- Cosine Similarity
    - this function will both calcuate and output results

In [17]:
# Print TF-IDF values in 'table' format
def print_TFIDF_for_all(terms, values, fileNames):
    print("\n======== TF-IDF Values =========\n")
    
    # Print headers
    header = ["Term"] + fileNames
    print("{:<20}".format(header[0]), end='')
    for name in header[1:]:
        print("{:<15}".format(name), end='')
    print("\n" + "-" * (20 + 15 * len(fileNames)))

    for term, tfidf_values in zip(terms, values):
        print("{:<20}".format(term), end='')
        for value in tfidf_values:
            print("{:<15.4f}".format(value), end='')
        print()

In [18]:
# Write TF-IDF values to a file in 'table' format
def write_TFIDF_for_all(terms, values, fileNames, outputFilePath):
    with open(outputFilePath, 'w') as f:
        f.write("Term")
        for name in fileNames:
            f.write("\t" + name)
        f.write("\n")

        for term, tfidf_values in zip(terms, values):
            f.write(term)
            for value in tfidf_values:
                f.write("\t{:.4f}".format(value))
            f.write("\n")


In [19]:
# Calculate and print cosine similarity for all pairs of documents
def calc_and_print_CosineSimilarity_for_all(tfs, fileNames):
    print("\n======== COSINE SIMILARITY =========\n")
    for i in range(len(fileNames)):
        print(f"File 1: {fileNames[i]}")
        for n in range(i + 1, len(fileNames)):
            similarity = cosine_similarity(tfs[i], tfs[n])[0][0]
            print(f"File 2: {fileNames[n]} - Cosine Similarity: {similarity:.4f}")
        print()

In [20]:
# Calculate and write cosine similarity to a file
def calc_and_write_CosineSimilarity_for_all(tfs, fileNames, outputFilePath):
    with open(outputFilePath, 'w') as f:
        f.write("File 1\tFile 2\tCosine Similarity\n")
        for i in range(len(fileNames)):
            for n in range(i + 1, len(fileNames)):
                similarity = cosine_similarity(tfs[i], tfs[n])[0][0]
                f.write(f"{fileNames[i]}\t{fileNames[n]}\t{similarity:.4f}\n")

## Wrap Everything into `Main()`

In [23]:
def main(printResults=True):
    baseFolderPath = "Cosine"

    fileNames, filePathList = returnListOfFilePaths(baseFolderPath)

    rawContentDict = create_docContentDict(filePathList)

    # calculate tfidf
    tfidf = TfidfVectorizer(tokenizer=processData, stop_words='english')
    tfs = tfidf.fit_transform(rawContentDict.values())
    tfs_Values = tfs.toarray()
    tfs_Term = tfidf.get_feature_names_out()

    if printResults:
        # print results
        print_TFIDF_for_all(tfs_Term, tfs_Values, fileNames)
        calc_and_print_CosineSimilarity_for_all(tfs, fileNames)
    else:
        # write results to file
        write_TFIDF_for_all(tfs_Term, tfs_Values, fileNames)
        calc_and_write_CosineSimilarity_for_all(tfs, fileNames)

In [24]:
if __name__ == "__main__":
    main(printResults=True)


======== TF-IDF Values =========

Term                f1.txt         f2.txt         f3.txt         
-----------------------------------------------------------------
'm                  0.0938         0.1108         0.0277         0.0000         0.0469         0.0469         0.0277         0.0277         0.0277         0.0000         0.0277         0.0000         0.0469         0.0277         0.0277         0.0357         0.0469         0.0277         0.0469         0.0277         0.0000         0.0277         0.0277         0.0938         0.0277         0.0000         0.0277         0.0000         0.0469         0.0554         0.0000         0.0000         0.0554         0.0469         0.0469         0.0554         0.0277         0.0277         0.0277         0.0469         0.0938         0.0277         0.0469         0.0831         0.0469         0.0554         0.0469         0.0469         0.0469         0.0469         0.1938         0.0469         0.0469         0.0000         0.1

D:\Anaconda\Lib\site-packages\sklearn\feature_extraction\text.py:525: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(
D:\Anaconda\Lib\site-packages\sklearn\feature_extraction\text.py:408: UserWarning: Your stop_words may be inconsistent with your preprocessing. Tokenizing the stop words generated tokens ['afterward', 'alon', 'alreadi', 'alway', 'anoth', 'anyon', 'anyth', 'anywher', 'becam', 'becom', 'besid', 'cri', 'describ', 'els', 'elsewher', 'empti', 'everi', 'everyon', 'everyth', 'everywher', 'fifti', 'formerli', 'forti', 'henc', 'hereaft', 'herebi', 'howev', 'hundr', 'inde', 'latterli', 'mani', 'meanwhil', 'moreov', 'mostli', 'nobodi', 'noon', 'noth', 'nowher', 'otherwis', 'perhap', 'pleas', 'seriou', 'sever', 'sinc', 'sincer', 'sixti', 'someon', 'someth', 'sometim', 'somewher', 'thenc', 'thereaft', 'therebi', 'therefor', 'thu', 'togeth', 'twelv', 'twenti', 'whatev', 'whenc', 'whenev', 'wherea', 'whereaft', 'wherebi', 'wh